# Day 39: Structured Output

AI gives great answers. Try parsing them in code — chaos.

Today we force the model to return **exact typed objects**. No JSON wrangling. No regex. Just Pydantic models in, Pydantic objects out.

## Install

In [1]:
%pip install langchain-google-genai pydantic --quiet


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup

In [ ]:
import os
import time
import json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

print("✅ Model ready")

## The Task

Same prompt for all methods: extract structured data from a job posting.

In [ ]:
JOB_POSTING = """
Senior Backend Engineer at TechNova (Bangalore)
Salary: ₹25-35 LPA | Remote-friendly | 3-5 years experience

We're looking for a backend engineer who knows Python, FastAPI, 
PostgreSQL, and has experience with microservices. Nice to have: 
Docker, Kubernetes, and AWS. Join our team of 40 engineers 
building India's fastest-growing fintech platform.

Apply by October 15, 2026.
"""

print("📄 Job posting loaded")

## Method 1: Prompt-Only ❌

Ask the model to return JSON. Hope for the best.

In [ ]:
start = time.time()

response = model.invoke(
    f"""Extract the following from this job posting and return as JSON:
- company, role, location, salary range, required skills, experience, deadline

Job posting:
{JOB_POSTING}"""
)

time_prompt = time.time() - start

print("⏱️", f"{time_prompt:.2f}s")
print()
print(response.content)

### The Problem

Sometimes you get clean JSON. Sometimes:
- Markdown fences wrapping the JSON
- Extra explanation text before/after
- Missing fields or inconsistent keys
- Booleans as strings, numbers as strings

**You can't `json.loads()` this reliably.**

In [ ]:
# Try parsing — this is what breaks in production
try:
    text = response.content.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    data = json.loads(text)
    print("✅ Parsed (got lucky this time)")
except json.JSONDecodeError as e:
    print(f"❌ Failed: {e}")
    print("   This is why prompt-only breaks in production")

## Method 2: with_structured_output ✅

One method. Pass a Pydantic model. Get a typed object back. Done.

In [ ]:
class JobPosting(BaseModel):
    """Structured job posting data."""
    company: str = Field(description="Company name")
    role: str = Field(description="Job title")
    location: str = Field(description="City/location")
    salary_min_lpa: float = Field(description="Minimum salary in LPA")
    salary_max_lpa: float = Field(description="Maximum salary in LPA")
    required_skills: list[str] = Field(description="Required technical skills")
    nice_to_have: list[str] = Field(description="Nice-to-have skills")
    experience_min: int = Field(description="Minimum years of experience")
    experience_max: int = Field(description="Maximum years of experience")
    remote_friendly: bool = Field(description="Whether remote work is allowed")
    team_size: int = Field(description="Number of people on the team")
    deadline: str = Field(description="Application deadline")

print("✅ Schema: 12 typed fields")

In [ ]:
structured_model = model.with_structured_output(JobPosting)

start = time.time()

job = structured_model.invoke(
    f"Extract all details from this job posting:\n{JOB_POSTING}"
)

time_structured = time.time() - start

print("⏱️", f"{time_structured:.2f}s")
print()
print(f"🏢 {job.company} — {job.role}")
print(f"📍 {job.location} ({'Remote OK' if job.remote_friendly else 'On-site'})")
print(f"💰 ₹{job.salary_min_lpa}-{job.salary_max_lpa} LPA")
print(f"🔧 Required: {', '.join(job.required_skills)}")
print(f"⭐ Nice to have: {', '.join(job.nice_to_have)}")
print(f"📅 {job.experience_min}-{job.experience_max} years | Team: {job.team_size}")
print(f"🗓️ Deadline: {job.deadline}")

### What Just Happened

```python
structured_model = model.with_structured_output(JobPosting)
job = structured_model.invoke(prompt)
```

That's it. Two lines. The response is a **Pydantic object** — not a string, not JSON, not something you need to parse. `job.company` is a `str`. `job.remote_friendly` is a `bool`. `job.required_skills` is a `list[str]`.

**Works with any LangChain model** — Gemini, OpenAI, Claude. Same code, same schema, any provider.

## Nested Schemas

Real data is nested. Structured output handles it.

In [ ]:
class Salary(BaseModel):
    """Salary information."""
    min_lpa: float = Field(description="Minimum salary in LPA")
    max_lpa: float = Field(description="Maximum salary in LPA")
    currency: str = Field(description="Currency code")

class Requirements(BaseModel):
    """Job requirements."""
    must_have: list[str] = Field(description="Required skills")
    nice_to_have: list[str] = Field(description="Optional skills")
    experience_years: str = Field(description="Experience range like '3-5 years'")

class Company(BaseModel):
    """Company details."""
    name: str
    team_size: int
    industry: str

class DetailedJob(BaseModel):
    """Complete job posting with nested structure."""
    role: str
    company: Company
    location: str
    remote: bool
    salary: Salary
    requirements: Requirements
    deadline: str

print("✅ Nested schema: 4 models, 15 fields")

In [ ]:
structured_nested = model.with_structured_output(DetailedJob)

start = time.time()

job = structured_nested.invoke(
    f"Extract all details from this job posting:\n{JOB_POSTING}"
)

time_nested = time.time() - start

print("⏱️", f"{time_nested:.2f}s")
print()
print(f"🏢 {job.company.name} ({job.company.industry})")
print(f"   Team: {job.company.team_size} engineers")
print(f"📍 {job.location} | Remote: {job.remote}")
print(f"💰 {job.salary.currency} {job.salary.min_lpa}-{job.salary.max_lpa} LPA")
print(f"🔧 Must have: {job.requirements.must_have}")
print(f"⭐ Nice to have: {job.requirements.nice_to_have}")
print(f"📅 {job.requirements.experience_years} | Deadline: {job.deadline}")

## Batch Extraction

Process multiple items in one call.

In [ ]:
class QuickJob(BaseModel):
    """Minimal job summary."""
    company: str
    role: str
    max_salary_lpa: float
    top_skill: str

class JobBatch(BaseModel):
    """Multiple extracted jobs."""
    jobs: list[QuickJob]

POSTINGS = """
1. Frontend Dev at Razorpay, Bangalore. ₹18-22 LPA. React, TypeScript.
2. ML Engineer at Swiggy, Remote. ₹30-40 LPA. Python, PyTorch, MLOps.
3. DevOps Lead at CRED, Bangalore. ₹35-45 LPA. Kubernetes, Terraform, AWS.
"""

structured_batch = model.with_structured_output(JobBatch)

start = time.time()
result = structured_batch.invoke(f"Extract all jobs:\n{POSTINGS}")
time_batch = time.time() - start

print(f"⏱️ {len(result.jobs)} jobs in {time_batch:.2f}s")
print()
for j in result.jobs:
    print(f"  {j.company:12} | {j.role:20} | ₹{j.max_salary_lpa} LPA | {j.top_skill}")

## Using It in a Chain

Structured output plugs into any LangChain pipeline.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

class Sentiment(BaseModel):
    """Sentiment analysis result."""
    sentiment: str = Field(description="positive, negative, or neutral")
    confidence: float = Field(description="Confidence score 0-1")
    key_phrase: str = Field(description="The phrase that most indicates the sentiment")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analyzer. Analyze the given text."),
    ("human", "{text}")
])

chain = prompt | model.with_structured_output(Sentiment)

# Run it
result = chain.invoke({"text": "This product is absolutely terrible. Waste of money."})
print(f"Sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence}")
print(f"Key phrase: '{result.key_phrase}'")

print()

result = chain.invoke({"text": "Best purchase I've made this year. Works perfectly!"})
print(f"Sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence}")
print(f"Key phrase: '{result.key_phrase}'")

## Comparison

In [ ]:
print("=" * 55)
print("METHOD COMPARISON")
print("=" * 55)
print()
print(f"{'Method':<28} {'Time':>7}  {'Reliable':>9}  {'Typed':>6}")
print("-" * 55)
print(f"{'Prompt-only':<28} {time_prompt:>6.2f}s  {'❌ No':>9}  {'❌ No':>6}")
print(f"{'with_structured_output':<28} {time_structured:>6.2f}s  {'✅ Yes':>9}  {'✅ Yes':>6}")
print(f"{'Nested schema':<28} {time_nested:>6.2f}s  {'✅ Yes':>9}  {'✅ Yes':>6}")
print(f"{'Batch (3 items)':<28} {time_batch:>6.2f}s  {'✅ Yes':>9}  {'✅ Yes':>6}")
print()
print("🏆 with_structured_output wins:")
print("   → Pydantic model in, typed object out")
print("   → Zero parsing code")
print("   → Works with any LangChain provider")
print("   → Nested + batch out of the box")

## Key Takeaways

1. **Prompt-only JSON is a trap** — works in demos, breaks in production
2. **`with_structured_output()` is the right way** — define a Pydantic model, get typed objects back, zero parsing
3. **It's provider-agnostic** — same code works with Gemini, OpenAI, Claude — just swap the model

Next: Day 40 — Streaming